# 00 · Synthetic data — checking notebook (dev only, temporary)

**What this is for.** Review the synthetic feature mart in one pass: is it a plausible retail book,
do the mechanics behave as specified, do the traps and the oracle work? The spec is
`docs/components/test_harness.md`; the review guide, with what "right" looks like for each section,
is `synthetic/REVIEW.md`. The guided EDA notebook replaces this one once EDA exists.

**How to use it.**
1. Generate the canonical dataset and its check tables once (about 6 minutes, plus about 10 for
   the checks): `python -m synthetic --preset canonical --out data/01_raw/synthetic_canonical --checks`.
2. Restart the kernel and run all cells (a few minutes).
3. Section 1 first: every check row must pass. A failing row is a generator bug, never a tolerance
   to widen. Then sections 2–10, with the review guide alongside.

The oracle columns are hidden truth for checking only; they are never features. Charts show
aggregates; most use a fixed 20% client sample to keep memory low.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

import synthetic
from synthetic.check_suite import read_checks
from synthetic.oracle import realised_target, window_probability
from synthetic.output import load_truth, read_output
from synthetic.traps import trap_summary

# The dataset lives next to the repo, never relative to the working directory.
DATA = Path(synthetic.__file__).resolve().parents[3] / "data" / "01_raw" / "synthetic_canonical"
truth = load_truth(str(DATA))  # fails loud if the dataset is older than the installed generator
spec = truth["spec"]
ORDER = list(spec["subsegments"])  # wealthiest first
PROFILES = ["private", "priority", "affluent", "mass"]
PROFILE_OF = {s: v["profile"] for s, v in spec["subsegments"].items()}
pd.set_option("display.max_columns", 50, "display.width", 200)
print(
    f"generator {truth['generator_version']} · seed {truth['seed']} · "
    f"{truth['n_clients']:,} clients"
    f" · {truth['n_months']} months from {truth['first_month']}"
)
print({k: v for k, v in truth["versions"].items()})

generator 0.3.0 · seed 0 · 100,000 clients · 24 months from 2024-10
{'numpy': '2.3.5', 'scipy': '1.16.2', 'pandas': '2.2.3', 'pyarrow': '19.0.1'}


## 1. Check tables
Every family's rows, from the spec, written independently of the generator, as saved by
`--checks` (each family runs in its own process: at 100k one family needs several GB). `failed`
must be 0 everywhere. The mart family reads every column, so it runs on the 20% client sample.

In [2]:
tables = read_checks(str(DATA))
summary = pd.DataFrame(
    [
        (name, len(t), int(t["passed"].sum()), int((~t["passed"]).sum()), int(t["rule"].sum()))
        for name, t in tables.items()
    ],
    columns=["family", "rows", "passed", "failed", "rules"],
)
display(summary)
failed = pd.concat([t[~t["passed"]].assign(family=name) for name, t in tables.items()])
failed if len(failed) else print("Every row passes.")

,family,rows,passed,failed,rules
0,population,158,158,0,7
1,holdings,378,378,0,52
2,holding_drivers,13,13,0,0
3,amount_rules,112,112,0,112
4,amount_levels,34,34,0,0
5,amount_mix,11,11,0,0
6,seasons,27,27,0,0
7,mart,81,81,0,81
8,traps,53,53,0,4
9,oracle,324,324,0,114


Every row passes.


## 2. Population
The mix should look like a UAE retail book: Mass dominant, Private small and old, ASIA dominant in
Mass, UAE nationals leading Private, digital onboarding more common lower down.

In [3]:
desc = read_output(
    str(DATA),
    "mart",
    [
        "client_id",
        "snapshot_month",
        "subsegment",
        "segment",
        "nationality_group",
        "employment_status",
        "onboarding_channel",
        "income",
        "age",
        "salary_transfer_flag",
        "relationship_days",
        "residence_days",
    ],
    client_sample=5,
)
first = desc[desc["snapshot_month"] == desc["snapshot_month"].min()]
first = first.assign(profile=first["subsegment"].map(PROFILE_OF))
display(first["subsegment"].value_counts(normalize=True).reindex(ORDER).round(4).to_frame("share"))
mix = first.groupby("profile")[
    ["nationality_group", "employment_status", "onboarding_channel"]
].agg(lambda s: s.value_counts(normalize=True).round(2).to_dict())
display(mix.reindex(PROFILES))
display(first.groupby("subsegment")["age"].quantile([0.1, 0.5, 0.9]).unstack().reindex(ORDER))
income = first.groupby("subsegment")["income"].agg(
    median="median", missing=lambda s: s.isna().mean()
)
display(income.reindex(ORDER).round(2))

,share
subsegment,
Upper Private,0.0020
Lower Private,0.0084
Upper Priority,0.0092
Lower Priority,0.0304
UA,0.0443
A,0.0421
UM,0.2032
MM,0.6165
LM,0.0440


,nationality_group,employment_status,onboarding_channel
profile,,,
private,"{'UAE': 0.38, 'ASIA': 0.17, 'WESTERN': 0.14, '...","{'Self-employed': 0.33, 'Employed': 0.25, 'Ret...","{'branch': 0.79, 'digital': 0.21}"
priority,"{'ASIA': 0.31, 'WESTERN': 0.3, 'UAE': 0.16, 'A...","{'Employed': 0.54, 'Self-employed': 0.24, 'Une...","{'branch': 0.69, 'digital': 0.31}"
affluent,"{'ASIA': 0.51, 'UAE': 0.14, 'ARAB': 0.13, 'WES...","{'Employed': 0.77, 'Self-employed': 0.12, 'Oth...","{'branch': 0.55, 'digital': 0.45}"
mass,"{'ASIA': 0.74, 'ARAB': 0.12, 'AFRICA': 0.07, '...","{'Employed': 0.91, 'Self-employed': 0.05, 'Oth...","{'digital': 0.55, 'branch': 0.45}"


,0.1,0.5,0.9
subsegment,,,
Upper Private,41.0,53.0,65.0
Lower Private,39.0,52.5,67.0
Upper Priority,33.7,46.0,59.0
Lower Priority,31.0,43.0,55.0
UA,29.0,41.0,53.0
A,28.0,39.0,51.7
UM,26.0,36.0,47.0
MM,24.0,34.0,45.0
LM,22.0,31.0,41.0


,median,missing
subsegment,,
Upper Private,128360.0,0.71
Lower Private,87450.0,0.53
Upper Priority,58005.0,0.22
Lower Priority,40820.0,0.19
UA,30860.0,0.11
A,20030.0,0.10
UM,12100.0,0.05
MM,6070.0,0.05
LM,3055.0,0.05


In [4]:
turnover = pd.DataFrame(truth["turnover"])
turnover = turnover.assign(
    exit_rate=turnover["exits"] / turnover["clients_before"],
    entry_rate=turnover["entries"] / turnover["clients_before"],
)
px.line(
    turnover.melt(["month", "profile"], ["exit_rate", "entry_rate"]),
    x="month",
    y="value",
    color="profile",
    line_dash="variable",
    title="Monthly entry and exit rates per profile",
).show()

## 3. Holdings and chains
Shares per profile should sit near the spec and stay flat (digital activity trends up by design:
the drift trap). Statuses split never / churned / active; loans mostly one account.

In [5]:
flags = [
    "casa_flag",
    "casa_active_flag",
    "cc_flag",
    "cc_active_flag",
    "td_flag",
    "wealth_digital_flag",
    "wealth_trad_flag",
    "ebanking_flag",
    "digital_active_flag",
    "fx_active_flag",
]
hold = read_output(
    str(DATA),
    "mart",
    [
        "client_id",
        "snapshot_month",
        "subsegment",
        *flags,
        "loan_personal_cnt",
        "loan_auto_cnt",
        "loan_home_cnt",
        "td_status",
        "product_cnt",
    ],
    client_sample=5,
)
hold = hold.assign(profile=hold["subsegment"].map(PROFILE_OF))
months = sorted(hold["snapshot_month"].unique())
ends = hold[hold["snapshot_month"].isin([months[0], months[-1]])]
loans = {f"{k}_cnt": lambda s: (s > 0).mean() for k in ["loan_personal", "loan_auto", "loan_home"]}
shares = ends.groupby(["profile", "snapshot_month"]).agg({**{f: "mean" for f in flags}, **loans})
display(shares.round(3))
monthly = hold.groupby(["snapshot_month", "profile"])[flags].mean().reset_index()
px.line(
    monthly.melt(["snapshot_month", "profile"]),
    x="snapshot_month",
    y="value",
    color="variable",
    facet_col="profile",
    title="Shares over the panel, per profile",
).show()
display(hold["td_status"].value_counts(normalize=True).round(3).to_frame("term deposit status"))

casa_flag  casa_active_flag  cc_flag  cc_active_flag  td_flag  wealth_digital_flag  wealth_trad_flag  ebanking_flag  digital_active_flag  fx_active_flag  loan_personal_cnt  \
profile  snapshot_month                                                                                                                                                                                
affluent 2024-10-01          0.992             0.930    0.647           0.507    0.158                0.143             0.104          0.892                0.708           0.305              0.208   
         2026-09-01          0.992             0.931    0.663           0.508    0.159                0.139             0.111          0.889                0.727           0.312              0.211   
mass     2024-10-01          0.990             0.953    0.299           0.216    0.044                0.021             0.009          0.744                0.590           0.120              0.202   
         2026-09-01          0.989             0.954    0.301           0.218    0.043                0.024             0.010          0.749                0.607           0.115              0.209   
priority 2024-10-01          0.986             0.920    0.778           0.641    0.308                0.181             0.346          0.890                0.719           0.478              0.127   
         2026-09-01          0.985             0.910    0.788           0.671    0.312                0.191             0.360          0.892                0.712           0.468              0.104   
private  2024-10-01          0.995             0.920    0.869           0.756    0.423                0.103             0.554          0.826                0.667           0.685              0.052   
         2026-09-01          0.996             0.928    0.830           0.704    0.300                0.090             0.561          0.834                0.709           0.601              0.022   

                         loan_auto_cnt  loan_home_cnt  
profile  snapshot_month                                
affluent 2024-10-01              0.135          0.114  
         2026-09-01              0.141          0.116  
mass     2024-10-01              0.060          0.021  
         2026-09-01              0.060          0.022  
priority 2024-10-01              0.078          0.213  
         2026-09-01              0.067          0.223  
private  2024-10-01              0.023          0.141  
         2026-09-01              0.022          0.126

,term deposit status
td_status,
never,0.839
churned,0.094
active,0.068


## 4. Amounts
Medians ordered by subsegment, heavy right tails, Private far above everyone else. Revolvers sit
at higher utilisation than transactors but rarely exactly at the limit.

In [6]:
amount_cols = [
    "casa_bal",
    "inflow_casa_vol",
    "outflow_casa_vol",
    "spend_cc_vol",
    "cc_limit",
    "cc_utilisation_ratio",
    "td_bal",
    "wealth_trad_bal",
    "wealth_digital_bal",
    "loan_personal_bal",
    "loan_home_bal",
    "fx_vol",
    "login_digital_mobile_cnt",
]
amt = read_output(
    str(DATA), "mart", ["client_id", "snapshot_month", "subsegment", *amount_cols], client_sample=5
)
may = amt[amt["snapshot_month"] == pd.Timestamp("2025-05-01").date()]
medians = may.groupby("subsegment")[amount_cols].agg(lambda s: s[s > 0].median()).reindex(ORDER)
display(medians.round(0))
counts, edges = np.histogram(np.log10(may.loc[may["casa_bal"] > 0, "casa_bal"]), bins=60)
px.bar(
    x=edges[:-1],
    y=counts,
    labels={"x": "log10 balance", "y": "clients"},
    title="Current/savings balance, May 2025 (20% sample)",
).show()
hidden = read_output(
    str(DATA), "oracle", ["client_id", "snapshot_month", "revolver_flag"], client_sample=5
)
cards = may.merge(hidden, on=["client_id", "snapshot_month"])
cards = cards[cards["cc_limit"] > 0].assign(
    kind=lambda d: np.where(d["revolver_flag"] == 1, "revolver", "transactor")
)
bins = np.linspace(0, 1, 21)
util = pd.DataFrame(
    {
        kind: np.histogram(part["cc_utilisation_ratio"], bins=bins)[0] / len(part)
        for kind, part in cards.groupby("kind")
    },
    index=bins[:-1].round(2),
)
px.bar(
    util, barmode="group", title="Card utilisation, revolvers vs transactors (share of cards)"
).show()

,casa_bal,inflow_casa_vol,outflow_casa_vol,spend_cc_vol,cc_limit,cc_utilisation_ratio,td_bal,wealth_trad_bal,wealth_digital_bal,loan_personal_bal,loan_home_bal,fx_vol,login_digital_mobile_cnt
subsegment,,,,,,,,,,,,,
Upper Private,3445335.0,157703.0,217253.0,71845.0,250000.0,0.0,2946621.0,9062609.0,890691.0,NaN,7054480.0,137980.0,6.0
Lower Private,2211901.0,136219.0,156441.0,29713.0,150000.0,0.0,1601167.0,6850650.0,734551.0,278385.0,4538069.0,81870.0,7.0
Upper Priority,144174.0,69270.0,72861.0,13920.0,100000.0,0.0,509418.0,1090331.0,214237.0,256482.0,3126902.0,36045.0,7.0
Lower Priority,100076.0,43070.0,42294.0,9959.0,75000.0,0.0,392215.0,645171.0,147017.0,155763.0,2233919.0,23887.0,8.0
UA,33113.0,35730.0,35303.0,6776.0,50000.0,0.0,180579.0,252755.0,70266.0,144013.0,1788284.0,13197.0,14.0
A,23671.0,22029.0,21803.0,4463.0,35000.0,0.0,118755.0,169903.0,39017.0,112421.0,1436040.0,8457.0,15.0
UM,5008.0,13084.0,13098.0,2885.0,10000.0,0.0,40352.0,46445.0,12410.0,65090.0,934225.0,3755.0,19.0
MM,2475.0,6411.0,6448.0,1505.0,10000.0,1.0,20376.0,30809.0,7221.0,38713.0,348822.0,2014.0,20.0
LM,1300.0,3149.0,3119.0,813.0,5000.0,0.0,12296.0,8555.0,4152.0,16562.0,137209.0,1107.0,20.0


In [7]:
flow_cols = [
    "inflow_casa_vol",
    "outflow_casa_vol",
    "casa_bal",
    "casa_active_flag",
    "outflow_casa_remittance_vol",
    "outflow_casa_cash_vol",
    "outflow_casa_bills_vol",
    "outflow_casa_transfers_vol",
    "spend_cc_travel_vol",
    "spend_cc_retail_vol",
    "spend_cc_online_vol",
    "spend_cc_dining_vol",
    "nationality_group",
]
flows = read_output(
    str(DATA), "mart", ["client_id", "snapshot_month", "subsegment", *flow_cols], client_sample=5
).sort_values(["client_id", "snapshot_month"])
# Rows with a previous row in the panel: a starting-book client's first panel row continues the
# burn-in.
change = flows["casa_bal"] - flows.groupby("client_id")["casa_bal"].shift()
gap = (flows["inflow_casa_vol"] - flows["outflow_casa_vol"] - change)[
    flows["casa_active_flag"] == 1
]
print(f"booking identity, largest |inflow − outflow − balance change|: {gap.abs().max():.4f}")
out_parts = [
    "outflow_casa_remittance_vol",
    "outflow_casa_cash_vol",
    "outflow_casa_bills_vol",
    "outflow_casa_transfers_vol",
]
card_parts = [
    "spend_cc_travel_vol",
    "spend_cc_retail_vol",
    "spend_cc_online_vol",
    "spend_cc_dining_vol",
]
prof = flows.assign(profile=flows["subsegment"].map(PROFILE_OF))
mix = pd.concat(
    {
        "outflow": prof.groupby("profile")[out_parts].sum(),
        "card spend": prof.groupby("profile")[card_parts].sum(),
    },
    axis=1,
)
display((mix["outflow"].div(mix["outflow"].sum(axis=1), axis=0)).reindex(PROFILES).round(3))
display((mix["card spend"].div(mix["card spend"].sum(axis=1), axis=0)).reindex(PROFILES).round(3))
display(
    flows.groupby("nationality_group")["outflow_casa_remittance_vol"]
    .agg(remitting=lambda s: (s > 0).mean(), mean_amount="mean")
    .round(2)
)

booking identity, largest |inflow − outflow − balance change|: 0.0000


,outflow_casa_remittance_vol,outflow_casa_cash_vol,outflow_casa_bills_vol,outflow_casa_transfers_vol
profile,,,,
private,0.002,0.007,0.036,0.955
priority,0.016,0.051,0.164,0.770
affluent,0.052,0.112,0.232,0.604
mass,0.134,0.253,0.227,0.387


,spend_cc_travel_vol,spend_cc_retail_vol,spend_cc_online_vol,spend_cc_dining_vol
profile,,,,
private,0.319,0.312,0.184,0.185
priority,0.182,0.370,0.241,0.207
affluent,0.084,0.435,0.286,0.196
mass,0.009,0.514,0.272,0.205


,remitting,mean_amount
nationality_group,,
AFRICA,0.55,1391.27
ARAB,0.55,1405.98
ASIA,0.57,1517.77
GCC,0.00,0.00
OTHER,0.54,1481.64
UAE,0.00,0.00
WESTERN,0.46,971.57


## 5. Time effects
Summer: travel spend and FX up, balances down, less so for Mass. School fees in September,
January and April. Bonuses January–April. Mobile logins jump at the app relaunch (2026-01).

In [8]:
seasonal = (
    prof.groupby(["snapshot_month", "profile"])
    .agg(
        travel=("spend_cc_travel_vol", "mean"),
        bills=("outflow_casa_bills_vol", "mean"),
        balance=("casa_bal", "median"),
    )
    .reset_index()
)
for column in ["travel", "bills", "balance"]:
    px.line(
        seasonal,
        x="snapshot_month",
        y=column,
        color="profile",
        log_y=True,
        title=f"{column} by month and profile",
    ).show()
logins = amt.groupby("snapshot_month")["login_digital_mobile_cnt"].apply(lambda s: s[s > 0].mean())
px.line(
    logins.reset_index(),
    x="snapshot_month",
    y="login_digital_mobile_cnt",
    title="Mean mobile logins of active users (drift at 2026-01)",
).show()

## 6. Segments overlap; Simpson's reversal
About a quarter of MM clients hold more than the UM median; Private far apart. Logins vs card
acquisition: positive within each onboarding channel, negative pooled.

In [9]:
mm_um = may[may["subsegment"].isin(["UM", "MM"]) & (may["casa_bal"] > 0)]
bins = np.linspace(1, 7, 61)
hist = pd.DataFrame(
    {
        s: np.histogram(np.log10(part["casa_bal"]), bins=bins)[0] / len(part)
        for s, part in mm_um.groupby("subsegment")
    },
    index=bins[:-1].round(1),
)
px.line(hist, title="log10 current/savings balance: UM vs MM (share of clients)").show()
um_median = mm_um.loc[mm_um["subsegment"] == "UM", "casa_bal"].median()
above = mm_um.loc[mm_um["subsegment"] == "MM", "casa_bal"] > um_median
print(f"share of MM above the UM median: {above.mean():.3f}")

share of MM above the UM median: 0.247


In [10]:
sim_cols = [
    "client_id",
    "snapshot_month",
    "cc_flag",
    "onboarding_channel",
    "subsegment",
    "relationship_days",
    "login_digital_mobile_cnt_0_2_months_avg",
]
sim_mart = read_output(str(DATA), "mart", None, client_sample=5)
sim_oracle = read_output(str(DATA), "oracle", None, client_sample=5)
chance = window_probability(sim_mart, sim_oracle, truth, "cc_acquire", 3)
frame = sim_mart[sim_cols].assign(chance=chance)
frame = frame[(frame["cc_flag"] == 0) & frame["chance"].notna()]
frame = frame.assign(
    logins=pd.qcut(
        frame["login_digital_mobile_cnt_0_2_months_avg"].rank(method="first"), 10, labels=False
    )
)
pooled = frame.groupby("logins")["chance"].mean().rename("pooled")
within = frame.groupby(["onboarding_channel", "logins"])["chance"].mean().unstack(0)
px.line(
    pd.concat([pooled, within], axis=1).reset_index(),
    x="logins",
    y=["pooled", "branch", "digital"],
    title="Card-acquisition chance by login decile",
).show()

## 7. Mart columns
145 columns from one grammar; nulls only where the retail data has them; rollups null until the
window fits inside the relationship.

In [11]:
schema = pd.DataFrame(truth["columns"])
display(schema.groupby("kind").size().to_frame("columns"))
nulls = sim_mart.isna().mean()
display(nulls[nulls > 0].sort_values(ascending=False).round(3).to_frame("null share"))
example = sim_mart[sim_mart["client_id"] == sim_mart["client_id"].iloc[1000]]
display(
    example[
        [
            "snapshot_month",
            "casa_bal",
            "casa_bal_0_2_months_avg",
            "casa_bal_0_5_months_avg",
            "casa_bal_0_5_months_std",
            "months_since_last_spend_cc",
            "relationship_days",
        ]
    ]
)

,columns
kind,
categorical,7
count,6
flow,20
holding,21
id,1
rollup,66
static,5
stock,10
time,1


,null share
legacy_risk_grade_trap_all_null,1.000
loan_home_maturity_months,0.961
loan_auto_maturity_months,0.934
loan_personal_maturity_months,0.800
months_since_last_fx,0.744
...,...
login_digital_mobile_cnt_0_2_months_avg,0.022
outflow_casa_vol_0_2_months_avg,0.022
fx_vol_0_2_months_avg,0.022
months_since_last_inflow_casa,0.017


,snapshot_month,casa_bal,casa_bal_0_2_months_avg,casa_bal_0_5_months_avg,casa_bal_0_5_months_std,months_since_last_spend_cc,relationship_days
1000,2024-10-01,3202.72,2986.693333,3046.198333,251.823512,<NA>,1325
21506,2024-11-01,3179.02,3194.376667,3041.863333,248.747563,<NA>,1355
42069,2024-12-01,2523.55,2968.430000,2959.291667,327.595279,<NA>,1386
62643,2025-01-01,2672.88,2791.816667,2889.255000,338.020561,<NA>,1417
83252,2025-02-01,3372.47,2856.300000,3025.338333,341.348884,<NA>,1445
103913,2025-03-01,3518.46,3187.936667,3078.183333,394.465341,<NA>,1476
124604,2025-04-01,3836.52,3575.816667,3183.816667,504.108964,<NA>,1506
145331,2025-05-01,3467.22,3607.400000,3231.850000,517.122901,<NA>,1537
166087,2025-06-01,2733.68,3345.806667,3266.871667,463.940908,<NA>,1567
186885,2025-07-01,4073.37,3424.756667,3500.286667,457.585821,<NA>,1598


## 8. Traps
The trap summary (planted traps end in `_trap_<kind>`), then the evidence: the leaks beat the
oracle, null controls sit at the bottom of every relevance ranking.

In [12]:
display(trap_summary(spec)[["product", "kind", "planted", "column", "expected"]])
leak_rows = tables["traps"]
display(
    leak_rows[leak_rows["check"].str.contains("leak|upstream|Ghana|Simpson|PSI", case=False)][
        ["check", "stratum", "expected", "realised", "tolerance", "passed"]
    ]
)

,product,kind,planted,column,expected
0,cc,leak,True,cc_preapproval_score_trap_leak,univariate AUC ≈ 0.97 for card acquisition in ...
1,cc,upstream_target,True,cc_acquire_next_3_months_flag_trap_upstream_ta...,equals Target Factory's own target except leav...
2,cc,simpson,False,login_digital_mobile_cnt,logins vs card acquisition: positive within ea...
3,cc,null_income_eligibility,False,income,a plain income filter silently drops exempt cl...
4,cc,onboarding,False,relationship_days,most adoption in the first 3 months: the stron...
5,cc,eligibility_defining,False,spend_cc_vol,"constant (0) within non-holders, the eligible ..."
6,cc,decimal_type,False,cc_limit,Spark reads a decimal; pandas gets Python obje...
7,loan_personal,process_leak,True,application_loan_personal_cnt_trap_process_leak,beats the oracle for openings in the next month
8,loan_personal,rare_level,False,nationality,stands out in event rate by level; unstable in...
9,loan_personal,inverted_u,False,age,"rank correlation ≈ 0, mutual information high"


,check,stratum,expected,realised,tolerance,passed
0,leak: AUC for cc_acquire,"w=3, one month",0.970000,0.970861,0.012487,True
1,leak: AUC above the oracle's,"w=3, one month",0.840738,0.970861,NaN,True
2,upstream: equals the realised cc_acquire (w=3),all,0.000000,0.000000,0.000000,True
3,upstream: 0 for leavers without a card,all,0.000000,0.000000,0.000000,True
4,upstream: null exactly for holders and the las...,all,0.000000,0.000000,0.000000,True
5,process leak: openings at t+1 with an applicat...,all,0.800000,0.796109,0.014057,True
6,process leak: applications at t declined (no o...,all,0.300000,0.300407,0.015097,True
7,process leak: AUC for loan_personal_open (w=1)...,all,0.755760,0.896873,NaN,True
8,rare level: opening rate Ghana / the rest of t...,realised above 2,2.000000,3.897985,NaN,True
9,rare level: opening rate Ghana / the rest of t...,realised vs the oracle's,4.071621,3.897985,1.133218,True


## 9. Oracle
The ceiling per target (oracle AUC, windows of 1, 3 and 6 months), calibration per decile for card
acquisition, regression coverage, and the relevance ranking's top and bottom.

In [13]:
targets = truth["oracle"]["targets"]
auc = pd.DataFrame(
    {
        t: {f"auc {w}": targets[t][w]["auc"] for w in ["w1", "w3", "w6"]}
        | {
            "positives in 6m": targets[t]["w3"]["expected_positives_6m"],
            "low confidence": targets[t]["w3"]["low_confidence"],
        }
        for t in targets
    }
).T
display(auc.round(3))
y = realised_target(sim_mart, "cc_acquire", 3)
cal = pd.DataFrame({"p": chance, "y": y}).dropna()
cal = cal.groupby(pd.qcut(cal["p"].rank(method="first"), 10, labels=False)).mean()
px.scatter(
    cal, x="p", y="y", title="cc_acquire (w=3): realised rate vs window probability per decile"
).show()
oracle_rows = tables["oracle"]
display(
    oracle_rows[oracle_rows["check"].str.contains("regression|realised /", case=False)][
        ["check", "stratum", "expected", "realised", "tolerance", "passed"]
    ].head(12)
)
ranking = pd.DataFrame(truth["oracle"]["relevance"]["cc_acquire"])
display(pd.concat([ranking.head(10), ranking.tail(5)]))

,auc w1,auc w3,auc w6,positives in 6m,low confidence
cc_acquire,0.862951,0.843224,0.831491,11115.167329,False
td_acquire,0.729588,0.726107,0.723974,5426.199632,False
wealth_digital_acquire,0.788729,0.785769,0.782326,7565.44595,False
wealth_trad_acquire,0.835984,0.830954,0.825858,851.476054,False
ebanking_acquire,0.949829,0.924318,0.898357,5006.639505,False
loan_personal_open,0.75576,0.754758,0.75578,12456.687124,False
loan_auto_open,0.706844,0.706849,0.706908,2988.232229,False
loan_home_open,0.793472,0.792609,0.793521,861.939994,False
fx_activate,0.746344,0.744318,0.741938,10334.002534,False
digital_activate,0.818264,0.786991,0.791146,30501.599141,False


,check,stratum,expected,realised,tolerance,passed
275,spend_cc_next_3_months: realised / predicted mean,observation 2024-11,1.0,1.001127,0.023541,True
279,spend_cc_next_3_months: realised / predicted mean,observation 2024-12,1.0,0.998849,0.023920,True
283,spend_cc_next_3_months: realised / predicted mean,observation 2025-06,1.0,0.996975,0.025785,True
287,spend_cc_next_3_months: realised / predicted mean,observation 2025-11,1.0,1.007420,0.023770,True
291,spend_cc_next_3_months: realised / predicted mean,observation 2025-12,1.0,1.005125,0.023273,True
295,spend_cc_next_3_months: realised / predicted mean,observation 2026-06,1.0,1.000953,0.024099,True
296,spend_cc_next_3_months: realised / predicted mean,predicted-mean decile 1,1.0,1.014725,0.061544,True
297,spend_cc_next_3_months: realised / predicted mean,predicted-mean decile 2,1.0,0.998212,0.042218,True
298,spend_cc_next_3_months: realised / predicted mean,predicted-mean decile 3,1.0,1.004719,0.034933,True
299,spend_cc_next_3_months: realised / predicted mean,predicted-mean decile 4,1.0,1.013532,0.029328,True


,column,score
0,income,0.545668
1,inflow_casa_vol_0_5_months_avg,0.516934
2,outflow_casa_vol_0_5_months_avg,0.516157
3,inflow_casa_vol_0_2_months_avg,0.508071
4,outflow_casa_vol_0_2_months_avg,0.507351
5,inflow_casa_vol_0_5_months_max,0.492866
6,inflow_casa_vol,0.478885
7,outflow_casa_vol,0.469037
8,outflow_casa_bills_vol_0_5_months_avg,0.461183
9,outflow_casa_bills_vol_0_2_months_avg,0.449700


## 10. Client trajectories and hidden traits
A few clients month by month: balances, spend, flags, and the hidden traits behind them.

In [14]:
some = sim_mart["client_id"].drop_duplicates().sample(3, random_state=0)
path = sim_mart[sim_mart["client_id"].isin(some)]
px.line(
    path, x="snapshot_month", y="casa_bal", color="client_id", log_y=True, title="Balances"
).show()
px.line(path, x="snapshot_month", y="spend_cc_vol", color="client_id", title="Card spend").show()
traits = sim_oracle[sim_oracle["client_id"].isin(some)].melt(
    ["client_id", "snapshot_month"], [c for c in sim_oracle.columns if c.startswith("z_")]
)
px.line(
    traits,
    x="snapshot_month",
    y="value",
    color="variable",
    facet_row="client_id",
    title="Hidden traits",
).show()